In [ ]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [ ]:
import os
# Download dataset from Kaggle using Kaggle API
# (Make sure kaggle.json is configured)

os.environ['KAGGLE_API_KEY'] = "XXX"
os.environ['KAGGLE_USERNAME'] = "XXX"

In [ ]:
# Requires Kaggle API setup (kaggle.json)
!kaggle datasets download -d maysee/mushrooms-classification-common-genuss-images

In [ ]:
!unzip mushrooms-classification-common-genuss-images.zip

In [ ]:
source_dir = "mushrooms/Mushrooms"

In [ ]:
import os
import shutil
import random

source_dir = "mushrooms/Mushrooms"
target_dir = "data"

os.makedirs(target_dir, exist_ok=True)

LIMIT = 200

for category in os.listdir(source_dir):
    src_path = os.path.join(source_dir, category)
    dst_path = os.path.join(target_dir, category)

    os.makedirs(dst_path, exist_ok=True)

    images = os.listdir(src_path)
    selected = random.sample(images, min(LIMIT, len(images)))

    for img in selected:
        shutil.copy(os.path.join(src_path, img),
                    os.path.join(dst_path, img))

print("Dataset reduced ")

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    "data",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_directory(
    "data",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

In [ ]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
output = layers.Dense(train_data.num_classes, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

In [ ]:
model.save("mushroom_model.h5")

In [ ]:
from tensorflow.keras.models import load_model

model = load_model("mushroom_model.h5")

In [ ]:
class_names = list(train_data.class_indices.keys())
print(class_names)

In [ ]:
from google.colab import files
files.download("mushroom_model.h5")